# V6.7-02 — YOLO26m LOO Ablation: M_124 (Modules 1 + 2 + 4, w/o Module 3)

Leave-One-Out (LOO) ablation training on `m_124`:
- **Module 1**: A (horizontal flip correction), B (reviewed missing box additions), F (tiny box removals)
- **Module 2**: C (person-as-FO drop), D (synthetic FO image purging)
- **Module 4**: H (SAM box tightening for insulator & foreign_object)
- *(Excluded Module 3: Quarantine E, Human review non-person drop G)*

Trained checkpoint, logs, and evaluation metrics are saved locally under `runs/curation_module_ablation/m_124` and mirrored to Camber at `stash://bunpmc/projects/finding_destiny/yolo26m_m_124`.

## 1. Setup

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from pathlib import Path
import json, os, subprocess, time
import yaml

def find_repo(start):
    for p in (start.resolve(), *start.resolve().parents):
        if (p / 'benchmakring_transmission_line_defect_model').is_dir(): return p
    raise FileNotFoundError('Repository root not found')

REPO = find_repo(Path.cwd())
BENCHMARK = REPO / 'benchmakring_transmission_line_defect_model'
DATA_ROOT = BENCHMARK / 'data/curation_module_ablation'
RUN_ROOT = BENCHMARK / 'runs/curation_module_ablation'
REMOTE_BASE = 'stash://bunpmc/projects/finding_destiny'

STAGE = 'm_124'
STAGE_INFO = {'modules': [1, 2, 4]}

candidates = (BENCHMARK/'weights/yolo26m.pt', BENCHMARK/'yolo26m.pt', REPO/'yolo26m.pt')
WEIGHTS = next((p for p in candidates if p.is_file()), Path('yolo26m.pt'))
print('Repository:', REPO)
print('Weights:', WEIGHTS)
print('Target Stage:', STAGE, STAGE_INFO)


Repository: D:\AI\Uav-Powerline-Defect-Detection
Weights: yolo26m.pt
Target Stage: m_124 {'modules': [1, 2, 4]}


## 2. Preflight dataset (read-only)

In [2]:
root = DATA_ROOT / STAGE
assert root.is_dir(), f"Dataset directory not found: {root}. Please run build_curation_loo_ablation.py first!"
cfg = yaml.safe_load((root/'data.yaml').read_text(encoding='utf-8'))
manifest = json.loads((root/'manifest.json').read_text(encoding='utf-8'))
assert manifest['modules'] == STAGE_INFO['modules'], f"Module mismatch: {manifest['modules']}"
assert cfg['val'] == 'images/val' and cfg['test'] == 'images/val_clean' and len(cfg['names']) == 6

counts = {}
for split in ('train', 'val', 'val_clean'):
    ni = len([p for p in (root/'images'/split).iterdir() if p.is_file() and p.suffix != '.cache'])
    nl = len(list((root/'labels'/split).glob('*.txt')))
    assert ni == nl, (STAGE, split, ni, nl)
    counts[split] = ni

print(STAGE, 'Modules:', manifest['modules'], 'Counts:', counts)

m_124 Modules: [1, 2, 4] Counts: {'train': 18052, 'val': 2272, 'val_clean': 2271}


## 3. Training, resume, and Camber helpers

Syncs every 5 epochs, on interruption, and after completion to Camber (`stash://bunpmc/projects/finding_destiny/`) and saves locally.

In [3]:
from ultralytics import YOLO

def load_camber_key():
    if os.environ.get('CAMBER_API_KEY'): return
    for line in (REPO/'.env').read_text(encoding='utf-8').splitlines():
        key, sep, value = line.partition('=')
        if sep and key.strip() == 'CAMBER_API_KEY' and value.strip():
            os.environ['CAMBER_API_KEY'] = value.strip().strip(chr(34)).strip(chr(39)); return
    raise RuntimeError('CAMBER_API_KEY missing from .env')

def wsl_path(path):
    path = Path(path).resolve()
    return f'/mnt/{path.drive.rstrip(chr(58)).lower()}{path.as_posix().split(chr(58),1)[1]}'

def camber_cp(local, remote, attempts=1):
    try:
        load_camber_key(); env = os.environ.copy()
        parts = [x for x in env.get('WSLENV','').split(':') if x]
        if 'CAMBER_API_KEY' not in parts: parts.append('CAMBER_API_KEY')
        env['WSLENV'] = ':'.join(parts)
        cmd = ['wsl','--','/home/dotru/.camber/bin/camber','stash','cp',wsl_path(local),remote]
        proc = subprocess.run(cmd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=5)
        if proc.returncode == 0: return
        print('Camber sync notice (non-fatal):', proc.stdout.strip()[:150], flush=True)
    except Exception as e:
        print('Camber sync skipped (local save active):', e, flush=True)

def paths(stage):
    return DATA_ROOT/stage, RUN_ROOT/stage/'yolo26m', f'{REMOTE_BASE}/yolo26m_{stage}'

def epochs_done(run_dir):
    p = run_dir/'results.csv'
    return max(0, len(p.read_text(encoding='utf-8').splitlines()) - 1) if p.is_file() else 0

def sync_run(run_dir, remote, trainer=None, final=False):
    files = [run_dir/'weights/last.pt', run_dir/'weights/best.pt', run_dir/'args.yaml', run_dir/'results.csv']
    if trainer is not None:
        p = run_dir/f'weights/epoch{trainer.epoch+1}.pt'
        if p.is_file(): files.append(p)
    if final:
        files = [p for p in run_dir.rglob('*') if p.is_file() and p.suffix != '.cache']
    for p in files:
        if p.is_file():
            camber_cp(p, f'{remote}/{p.relative_to(run_dir).as_posix()}')

def train_stage(stage, batch_size=32):
    data_root, run_dir, remote = paths(stage)
    data, manifest = data_root/'data.yaml', data_root/'manifest.json'
    camber_cp(data, f'{remote}/data.yaml')
    camber_cp(manifest, f'{remote}/dataset_manifest.json')

    last_pt, best_pt = run_dir/'weights/last.pt', run_dir/'weights/best.pt'
    if epochs_done(run_dir) >= 50 and best_pt.is_file():
        print(f'[SKIP] {stage} completed'); sync_run(run_dir, remote, final=True); return None

    resume = last_pt.is_file()
    model = YOLO(str(last_pt if resume else WEIGHTS))

    def on_save(trainer):
        epoch = trainer.epoch + 1
        if epoch % 5 == 0 or epoch == trainer.epochs:
            try: sync_run(run_dir, remote, trainer=trainer)
            except Exception as exc: print('Camber sync warning:', exc, flush=True)

    model.add_callback('on_model_save', on_save)
    print(f'[{stage}] {"RESUME" if resume else "START"} -> {run_dir} (batch={batch_size})')
    try:
        if resume:
            result = model.train(resume=True)
        else:
            result = model.train(
                data=str(data),
                project=str(RUN_ROOT/stage),
                name='yolo26m',
                exist_ok=True,
                epochs=50,
                imgsz=640,
                batch=batch_size,
                device=0,
                workers=0,
                optimizer='AdamW',
                lr0=0.001,
                lrf=0.01,
                cos_lr=True,
                warmup_epochs=3.0,
                weight_decay=0.0005,
                close_mosaic=10,
                amp=True,
                seed=42,
                deterministic=True,
                pretrained=True,
                val=True,
                save=True,
                save_period=5,
                plots=True,
                cache=False
            )
    except BaseException:
        sync_run(run_dir, remote)
        raise
    sync_run(run_dir, remote, final=True)
    print(f'[{stage}] complete -> Local: {run_dir} | Camber: {remote}')
    return result


## 4. Train M_124

> **Lưu ý VRAM:** Nếu `batch=32` bị CUDA Out-of-Memory trên GPU RTX 4050 6GB, hãy đổi sang `batch=8` hoặc `batch=4`.

In [4]:
# Khởi chạy huấn luyện
m124_result = train_stage(STAGE, batch_size=8)


Camber sync notice (non-fatal): /bin/sh: /home/dotru/.camber/bin/camber: not found
Camber sync notice (non-fatal): /bin/sh: /home/dotru/.camber/bin/camber: not found
[m_124] START -> D:\AI\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\runs\curation_module_ablation\m_124\yolo26m (batch=8)
New https://pypi.org/project/ultralytics/8.4.208 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.172  Python-3.13.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\AI\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\curation_module_ablation\m_124\data.yaml, degrees=0.0, det

KeyboardInterrupt: 

## 5. Status

In [ ]:
_, run_dir, remote = paths(STAGE)
print(f'{STAGE:10} epochs={epochs_done(run_dir):2}/50 last={(run_dir/"weights/last.pt").is_file()} best={(run_dir/"weights/best.pt").is_file()} {remote}')

## 6. Held-out test evaluation

Evaluates `best.pt` on frozen `test = val_clean`.

In [ ]:
data_root, run_dir, _ = paths(STAGE)
best = run_dir/'weights/best.pt'
if best.is_file():
    test_result = YOLO(str(best)).val(
        data=str(data_root/'data.yaml'),
        split='test',
        imgsz=640,
        batch=8,
        device=0,
        workers=0,
        project=str(run_dir),
        name='test',
        plots=True
    )
else:
    print(f'[SKIP TEST] {STAGE}: best.pt missing')
